In [1]:
import numpy as np
import pandas as pd

In [2]:
station = "ashok_vihar"

In [3]:
from training.train import load_station_df

df, ts, target_idx = load_station_df(station)

In [4]:
df.shape

(43824, 16)

In [5]:
instances = df.shape[0] - 4 - 1 + 1

tr_instances = int(instances * 0.7)
vl_instances = int(instances * 0.15)

values = df.to_numpy()

train_rows = values[: tr_instances + 4 + 1 - 1]  # Combinations
vmin = np.min(train_rows, axis=0)
vmax = np.max(train_rows, axis=0)
scale = np.where(vmax - vmin == 0, 1.0, vmax - vmin)

scaled = (values - vmin) / scale
scaled

array([[0.7076749 , 0.7467304 , 0.3429613 , ..., 0.        , 0.00175408,
        0.9820684 ],
       [0.7646727 , 0.8078471 , 0.3894673 , ..., 0.        , 0.00149681,
        0.9820684 ],
       [0.7765237 , 0.8332495 , 0.4148268 , ..., 0.        , 0.00177747,
        0.98192954],
       ...,
       [0.3295711 , 0.46579477, 0.02325299, ..., 0.        , 0.01157692,
        0.9308684 ],
       [0.33069977, 0.46981892, 0.02471136, ..., 0.        , 0.01157692,
        0.9308684 ],
       [0.30812642, 0.43661973, 0.00945919, ..., 0.        , 0.01157692,
        0.9308684 ]], shape=(43824, 16), dtype=float32)

In [6]:
len(scaled)

43824

In [7]:
from training.train import make_windows, time_codes

X, Y, TC = make_windows(scaled, target_idx, 4, 1, time_codes(ts))

In [8]:
len(values)

43824

In [9]:
scale

array([886.     , 994.     , 493.69998, 338.73   , 494.9    , 249.27501,
       184.2    ,   9.73   , 199.8    ,  39.32   ,  76.76667,   5.25   ,
       359.34998,  21.28   , 855.15   , 215.82   ], dtype=float32)

In [10]:
Y.shape

(43820, 1, 2)

In [11]:
TC.shape

(43820, 4, 3)

In [12]:


sl = {
        "train": slice(0, tr_instances),
        "val": slice(tr_instances, tr_instances + vl_instances),
        "test": slice(tr_instances + vl_instances, instances)
    }

In [13]:
import torch

data = {k: (torch.from_numpy(X[s]), torch.from_numpy(Y[s]), torch.from_numpy(TC[s])) for k, s in sl.items()}
meta = dict(target_idx=target_idx, in_dim=X.shape[-1], num_nodes=X.shape[2], y_min=vmin[target_idx],
                y_scale=scale[target_idx], timestamps=ts.values, test_start=sl["test"].start)

In [14]:
data

{'train': (tensor([[[0.7077, 0.7467, 0.3430,  ..., 0.0000, 0.0018, 0.9821],
           [0.7647, 0.8078, 0.3895,  ..., 0.0000, 0.0015, 0.9821],
           [0.7765, 0.8332, 0.4148,  ..., 0.0000, 0.0018, 0.9819],
           [0.7585, 0.8169, 0.5418,  ..., 0.0000, 0.0018, 0.9815]],
  
          [[0.7647, 0.8078, 0.3895,  ..., 0.0000, 0.0015, 0.9821],
           [0.7765, 0.8332, 0.4148,  ..., 0.0000, 0.0018, 0.9819],
           [0.7585, 0.8169, 0.5418,  ..., 0.0000, 0.0018, 0.9815],
           [0.7460, 0.7890, 0.5142,  ..., 0.0000, 0.0016, 0.9817]],
  
          [[0.7765, 0.8332, 0.4148,  ..., 0.0000, 0.0018, 0.9819],
           [0.7585, 0.8169, 0.5418,  ..., 0.0000, 0.0018, 0.9815],
           [0.7460, 0.7890, 0.5142,  ..., 0.0000, 0.0016, 0.9817],
           [0.7427, 0.7980, 0.5639,  ..., 0.0000, 0.0018, 0.9830]],
  
          ...,
  
          [[0.0327, 0.0835, 0.0035,  ..., 0.0000, 0.0092, 0.9216],
           [0.0361, 0.0845, 0.0030,  ..., 0.0000, 0.0092, 0.9216],
           [0.0372, 0.0

In [15]:
meta

{'target_idx': [0, 1],
 'in_dim': 16,
 'num_nodes': 16,
 'y_min': array([1., 2.], dtype=float32),
 'y_scale': array([886., 994.], dtype=float32),
 'timestamps': array(['2021-01-01T00:00:00.000000', '2021-01-01T01:00:00.000000',
        '2021-01-01T02:00:00.000000', ..., '2025-12-31T21:00:00.000000',
        '2025-12-31T22:00:00.000000', '2025-12-31T23:00:00.000000'],
       shape=(43824,), dtype='datetime64[us]'),
 'test_start': 37246}

In [16]:
df, ts, target_idx = load_station_df("ashok_vihar")
pd.to_datetime(np.asarray(ts))

DatetimeIndex(['2021-01-01 00:00:00', '2021-01-01 01:00:00',
               '2021-01-01 02:00:00', '2021-01-01 03:00:00',
               '2021-01-01 04:00:00', '2021-01-01 05:00:00',
               '2021-01-01 06:00:00', '2021-01-01 07:00:00',
               '2021-01-01 08:00:00', '2021-01-01 09:00:00',
               ...
               '2025-12-31 14:00:00', '2025-12-31 15:00:00',
               '2025-12-31 16:00:00', '2025-12-31 17:00:00',
               '2025-12-31 18:00:00', '2025-12-31 19:00:00',
               '2025-12-31 20:00:00', '2025-12-31 21:00:00',
               '2025-12-31 22:00:00', '2025-12-31 23:00:00'],
              dtype='datetime64[us]', length=43824, freq=None)

In [128]:
from torch.utils.data import DataLoader, TensorDataset

mk = lambda k: DataLoader(TensorDataset(*data[k][:3]), batch_size=32, shuffle=False, num_workers=0)

train_dl, val_dl = mk("train"), mk("val")

In [129]:
batch = next(iter(train_dl))
x, y, tc = batch[0], batch[1], batch[2]

print(x.shape, y.shape, tc.shape)

torch.Size([32, 4, 16]) torch.Size([32, 1, 2]) torch.Size([32, 4, 3])


In [130]:
X_val, Y_val, TC_val = data["val"]
Y_val

tensor([[[0.0564, 0.0674]],

        [[0.0508, 0.0744]],

        [[0.0801, 0.1046]],

        ...,

        [[0.0824, 0.2052]],

        [[0.0937, 0.1962]],

        [[0.1332, 0.2173]]])

In [131]:
# instances = df.shape[0] - 4 - 1 + 1
# tr_instances = int(instances * .7)
#
# train_rows = df.iloc[: tr_instances]  # Combinations
# vmin = np.min(train_rows, axis=0)

In [132]:
# train_rows.tail()

In [133]:
data = np.load(r"C:\Users\VISHAL\Desktop\CPS Internship Work\AQI Forecast\aqi_research\results\predictions\lstm_ashok_vihar_T4_H1_s42.npz")

In [134]:
data.files

['observed', 'predicted']

In [135]:
y = data['observed']
p = data['predicted']

In [141]:
y * scale[[0,1]] + vmin[[0,1]] - p

array([[[ 14.701027 , -10.367325 ]],

       [[ 24.083176 ,   4.0594482]],

       [[ 36.151623 ,   3.8785706]],

       ...,

       [[121.606674 ,  76.58325  ]],

       [[104.03258  ,  56.221893 ]],

       [[ 84.862915 ,  20.531128 ]]], shape=(6574, 1, 2), dtype=float32)